# 压缩空气储能仿真

本 Notebook 是一个面向代码可视化展示的 **先进绝热压缩空气储能 AA-CAES** 系统级 process 仿真原型。它把压缩机、换热器、储气罐、热储能 TES、膨胀机和发电机放在同一个动态流程中，展示电能、压力能和热能在完整充放电周期中的转化。

流程链路：

```text
环境空气 -> 多级压缩 -> 级间冷却与压缩热回收 -> 高压储气罐
       -> 静置保压/热损失
       -> TES 加热 -> 多级膨胀 -> 发电机输出
```


## 1. 工艺流程说明

本模型用于论文题目“基于代码可视化的压缩空气储能原理仿真系统设计与实现”中的 CAES 原理仿真案例。它不是三维 CFD，也不是工业厂商级设备选型软件，而是一个适合 Notebook 可视化、参数绑定和原理教学的系统级动态 process 模型。

系统包含三个阶段：

1. **充电阶段**：电动机驱动多级压缩机，空气被压缩后进入高压储气罐，压缩热通过换热器进入 TES。
2. **静置阶段**：储气罐和 TES 与环境发生热交换，压力、温度和储热量缓慢变化。
3. **放电阶段**：高压空气从储气罐释放，经 TES 加热后进入多级膨胀机，带动发电机输出电功率。


## 2. 建模假设

1. 采用 0D 集中参数模型描述储气罐，不做三维空间温度场。
2. 空气物性优先使用 CoolProp；若当前环境未安装 CoolProp，则使用可见的变比热 `cp(T)` 备用模型，便于先运行和检查模型结构。
3. 压缩机和膨胀机采用多级等熵效率模型，各级按等压比分配。
4. 换热器采用有效度模型，压缩热进入 TES，放电时 TES 为膨胀前空气再热。
5. TES 采用集中参数储热模型，用储热量和等效温度描述充热、放热和热损失。
6. 管路压降、阀门节流和真实设备 map 暂不纳入当前原型，后续可作为模型增强项。
7. 时间推进采用显式欧拉法，目的是清晰展示状态变量如何随时间更新。


## 3. 参数说明表

| 参数 | 含义 | 默认值 |
| --- | --- | --- |
| `operation_profile` | 运行工况 | `charge_hold_discharge` |
| `ambient_temperature_c` | 环境温度 | 25 degC |
| `ambient_pressure_bar` | 环境压力 | 1.01325 bar |
| `compressor_stages` | 压缩机级数 | 3 |
| `expander_stages` | 膨胀机级数 | 2 |
| `mass_flow_kg_s` | 空气质量流量 | 12 kg/s |
| `compressor_efficiency` | 压缩机等熵效率 | 0.82 |
| `expander_efficiency` | 膨胀机等熵效率 | 0.86 |
| `heat_exchanger_effectiveness` | 换热器有效度 | 0.88 |
| `storage_volume_m3` | 储气罐容积 | 1200 m3 |
| `min_storage_pressure_bar` | 最低放电压力 | 40 bar |
| `max_storage_pressure_bar` | 最高储气压力 | 120 bar |
| `tes_mass_kg` | TES 储热介质质量 | 420000 kg |
| `tes_initial_temperature_c` | TES 初始温度 | 120 degC |
| `charge_hours` / `hold_hours` / `discharge_hours` | 充电/静置/放电时长 | 4 / 2 / 4 h |
| `time_step_minutes` | 时间步长 | 2 min |


## 4. 数学模型与控制方程

### 多级压缩机

每一级压缩机按等熵效率计算：

```text
h_out = h_in + (h_out,s - h_in) / eta_c
W_c = m_dot * (h_out - h_in)
```

### 储气罐动态模型

储气罐使用质量守恒和能量守恒：

```text
dm/dt = m_in - m_out
U_new = U_old + m_in h_in dt - m_out h_out dt + Q_wall dt
```

其中 `Q_wall = -UA_storage * (T_tank - T_ambient)`。

### TES 储热模型

```text
E_TES,new = E_TES,old + Q_charge dt - Q_discharge dt - UA_TES (T_TES - T_ambient) dt
T_TES = T_ambient + E_TES / (m_TES cp_TES)
```

### 多级膨胀机

```text
h_out = h_in - eta_t * (h_in - h_out,s)
W_t = m_dot * (h_in - h_out)
```

### 往返效率

```text
eta_rt = electric_output / electric_input
```


## 5. 计算环境

这里导入 Notebook 后续计算和绘图所需的基础库。


In [ ]:
# 计算环境
%matplotlib inline
import math
import warnings
import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    def display(value):
        print(value)

plt.rcParams['font.sans-serif'] = [
    'SimHei',
    'WenQuanYi Micro Hei',
    'WenQuanYi Zen Hei',
    'Microsoft YaHei',
    'Arial Unicode MS',
    'sans-serif'
]
plt.rcParams['axes.unicode_minus'] = False

def markdown_table(rows, columns):
    header = '| ' + ' | '.join(columns) + ' |'
    separator = '| ' + ' | '.join(['---'] * len(columns)) + ' |'
    body = []
    for row in rows:
        body.append('| ' + ' | '.join(str(row.get(column, '')) for column in columns) + ' |')
    return '\n'.join([header, separator, *body])


## 6. 参数层代码

这里定义 CAES process 仿真的主要输入参数。后续集成到系统后，这一层会被参数绑定侧边栏直接识别和修改。


In [ ]:
# 参数层代码
operation_profile = 'charge_hold_discharge'

# 环境参数
ambient_temperature_c = 25.0
ambient_pressure_bar = 1.01325

# 压缩/膨胀设备
compressor_stages = 3
expander_stages = 2
mass_flow_kg_s = 12.0
compressor_efficiency = 0.82
expander_efficiency = 0.86
motor_efficiency = 0.96
generator_efficiency = 0.96

# 换热与储气
heat_exchanger_effectiveness = 0.88
storage_volume_m3 = 1200.0
min_storage_pressure_bar = 40.0
max_storage_pressure_bar = 120.0
initial_storage_pressure_bar = 45.0
storage_heat_transfer_coefficient_wk = 2800.0

# 热储能 TES
tes_mass_kg = 420000.0
tes_specific_heat_kj_kg_k = 0.92
tes_initial_temperature_c = 120.0
tes_ambient_loss_coefficient_wk = 1800.0
max_turbine_inlet_temperature_c = 520.0
minimum_tes_approach_temperature_k = 12.0

# 运行周期
charge_hours = 4.0
hold_hours = 2.0
discharge_hours = 4.0
time_step_minutes = 2.0


## 7. 物性层代码

本层优先使用 CoolProp 计算空气物性。如果当前环境没有 CoolProp，Notebook 会启用可见的 `cp(T)` 变比热备用模型。备用模型的目的只是让原型文件可直接运行；正式系统集成时建议在 Docker 镜像中安装 CoolProp。


In [ ]:
# 物性层代码
AIR = 'Air'
R_AIR = 287.05
T_REF = 298.15
P_REF = 101325.0

try:
    import CoolProp.CoolProp as CP
    COOLPROP_AVAILABLE = True
except ImportError:
    CP = None
    COOLPROP_AVAILABLE = False
    warnings.warn(
        '当前环境未安装 CoolProp，已启用透明 cp(T) 备用模型。正式系统建议安装: pip install CoolProp',
        RuntimeWarning
    )

def cp_air_polynomial(T):
    """Variable cp(T) fallback for dry air, J/(kg*K), valid for principle-level simulation."""
    theta = np.asarray(T, dtype=float) - 300.0
    cp = 1006.0 + 0.085 * theta + 1.2e-4 * theta**2
    return np.maximum(cp, 950.0)

def h_air_fallback(T):
    """Analytical integral of the fallback cp(T), referenced to 273.15 K."""
    T = np.asarray(T, dtype=float)
    x = T - 300.0
    x0 = 273.15 - 300.0
    return (
        1006.0 * (T - 273.15)
        + 0.085 * (x**2 - x0**2) / 2
        + 1.2e-4 * (x**3 - x0**3) / 3
    )

def T_from_h_fallback(h_target, low=180.0, high=1200.0):
    """Invert fallback h(T) using bisection."""
    lo, hi = low, high
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if h_air_fallback(mid) < h_target:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)

def T_from_u_rho_fallback(u_target, rho, low=180.0, high=1200.0):
    """Invert u(T)=h(T)-R*T for the fallback model."""
    lo, hi = low, high
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        u_mid = h_air_fallback(mid) - R_AIR * mid
        if u_mid < u_target:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)

def air_props(T, p):
    """Return air properties at temperature T [K] and pressure p [Pa]."""
    if COOLPROP_AVAILABLE:
        return {
            'T': T,
            'p': p,
            'cp': CP.PropsSI('Cpmass', 'T', T, 'P', p, AIR),
            'cv': CP.PropsSI('Cvmass', 'T', T, 'P', p, AIR),
            'h': CP.PropsSI('Hmass', 'T', T, 'P', p, AIR),
            'u': CP.PropsSI('Umass', 'T', T, 'P', p, AIR),
            's': CP.PropsSI('Smass', 'T', T, 'P', p, AIR),
            'rho': CP.PropsSI('Dmass', 'T', T, 'P', p, AIR)
        }

    cp = float(cp_air_polynomial(T))
    cv = cp - R_AIR
    h = float(h_air_fallback(T))
    rho = p / (R_AIR * T)
    s = cp * math.log(T / T_REF) - R_AIR * math.log(p / P_REF)
    return {'T': T, 'p': p, 'cp': cp, 'cv': cv, 'h': h, 'u': h - R_AIR * T, 's': s, 'rho': rho}

def h_from_T_p(T, p):
    return air_props(T, p)['h']

def u_from_T_p(T, p):
    return air_props(T, p)['u']

def rho_from_T_p(T, p):
    return air_props(T, p)['rho']

def T_from_h_p(h, p):
    if COOLPROP_AVAILABLE:
        return CP.PropsSI('T', 'P', p, 'Hmass', h, AIR)
    return T_from_h_fallback(h)

def T_p_from_u_rho(u, rho):
    if COOLPROP_AVAILABLE:
        try:
            T = CP.PropsSI('T', 'Umass', u, 'Dmass', rho, AIR)
            p = CP.PropsSI('P', 'T', T, 'Dmass', rho, AIR)
            return T, p
        except Exception:
            pass
    T = T_from_u_rho_fallback(u, rho)
    p = rho * R_AIR * T
    return T, p

def isentropic_h_out_compression(T_in, p_in, p_out):
    if COOLPROP_AVAILABLE:
        s_in = CP.PropsSI('Smass', 'T', T_in, 'P', p_in, AIR)
        return CP.PropsSI('Hmass', 'P', p_out, 'Smass', s_in, AIR)
    props = air_props(T_in, p_in)
    gamma = props['cp'] / props['cv']
    T_out_s = T_in * (p_out / p_in) ** ((gamma - 1) / gamma)
    return h_air_fallback(T_out_s)

def isentropic_h_out_expansion(T_in, p_in, p_out):
    return isentropic_h_out_compression(T_in, p_in, p_out)

print('CoolProp 可用:', COOLPROP_AVAILABLE)
print('环境空气 cp:', f"{air_props(298.15, 101325)['cp']:.2f} J/(kg*K)")


## 8. 设备模型层代码

本层按设备边界拆分为六个小模型，代码中也用分段标题标出：

1. **单位换算与公共常量**：把参数层输入转成计算用 SI 单位。
2. **压缩机单级模型**：输入入口温度、入口压力、出口压力、质量流量和等熵效率，输出出口温度、出口焓和压缩功率。
3. **膨胀机单级模型**：输入入口温度、入口压力、出口压力、质量流量和等熵效率，输出出口温度、出口焓和膨胀功率。
4. **冷却器/换热器模型**：根据换热器有效度计算冷却后温度和换热量。
5. **定容储气罐动态模型**：根据质量守恒和能量守恒更新 `m_tank`、`T_tank` 和 `p_tank`。
6. **热储能 TES 动态模型**：根据充热、放热和热损失更新储热量和等效温度。


In [ ]:
# 设备模型层代码
# ============================================================
# 单位换算与公共常量
# ------------------------------------------------------------
# 作用：把参数层中便于用户理解的单位，转换成模型计算使用的 SI 单位。
# 注意：后续所有设备模型默认使用 K、Pa、kg/s、J、W。
# ============================================================
ambient_temperature_k = ambient_temperature_c + 273.15
ambient_pressure_pa = ambient_pressure_bar * 1e5
tes_specific_heat_j_kg_k = tes_specific_heat_kj_kg_k * 1000

# ============================================================
# 模型 1：压缩机单级模型 compressor_stage
# ------------------------------------------------------------
# 物理意义：把空气从 p_in 压缩到 p_out。
# 输入：T_in [K], p_in [Pa], p_out [Pa], eta_isentropic [-], mass_flow [kg/s]
# 输出：T_out [K], h_out [J/kg], power_w [W]
# 原理级公式：
#   T2 = T1 * (1 + (pi_c^((k-1)/k) - 1) / eta_c)
#   Wc = mdot * cp * (T2 - T1)
# 当前实现：优先用焓差 h_out - h_in；无 CoolProp 时由 cp(T) 备用模型给出焓。
# ============================================================
def compressor_stage(T_in, p_in, p_out, eta_isentropic, mass_flow):
    h_in = h_from_T_p(T_in, p_in)
    h_out_s = isentropic_h_out_compression(T_in, p_in, p_out)
    h_out = h_in + (h_out_s - h_in) / eta_isentropic
    T_out = T_from_h_p(h_out, p_out)
    power_w = mass_flow * (h_out - h_in)
    return T_out, h_out, power_w

# ============================================================
# 模型 2：膨胀机单级模型 expander_stage
# ------------------------------------------------------------
# 物理意义：高压热空气从 p_in 膨胀到 p_out，并输出轴功。
# 输入：T_in [K], p_in [Pa], p_out [Pa], eta_isentropic [-], mass_flow [kg/s]
# 输出：T_out [K], h_out [J/kg], power_w [W]
# 原理级公式：
#   T4 = T3 * (1 - eta_t * (1 - pi_t^((1-k)/k)))
#   Wt = mdot * cp * (T3 - T4)
# 当前实现：用等熵焓降乘以膨胀机等熵效率。
# ============================================================
def expander_stage(T_in, p_in, p_out, eta_isentropic, mass_flow):
    h_in = h_from_T_p(T_in, p_in)
    h_out_s = isentropic_h_out_expansion(T_in, p_in, p_out)
    h_out = h_in - eta_isentropic * (h_in - h_out_s)
    T_out = T_from_h_p(h_out, p_out)
    power_w = mass_flow * max(h_in - h_out, 0.0)
    return T_out, h_out, power_w

# ============================================================
# 模型 3：冷却器 / 换热器有效度模型 cooler
# ------------------------------------------------------------
# 物理意义：压缩后的高温空气向 TES 或冷端介质放热。
# 输入：热端入口温度、冷端入口温度、两侧热容率、换热器有效度。
# 输出：热端出口温度、冷端出口温度、换热功率。
# 原理级关系：Q = epsilon * C_min * (T_hot,in - T_cold,in)
# ============================================================
def cooler(T_hot_in, T_cold_in, heat_capacity_hot, heat_capacity_cold, effectiveness):
    """epsilon heat exchanger model. Heat capacity rates use W/K."""
    delta_t = max(T_hot_in - T_cold_in, 0.0)
    if delta_t <= 0 or heat_capacity_hot <= 0 or heat_capacity_cold <= 0:
        return T_hot_in, T_cold_in, 0.0
    c_min = min(heat_capacity_hot, heat_capacity_cold)
    heat_w = effectiveness * c_min * delta_t
    T_hot_out = T_hot_in - heat_w / heat_capacity_hot
    T_cold_out = T_cold_in + heat_w / heat_capacity_cold
    return T_hot_out, T_cold_out, heat_w

# ============================================================
# 模型 4：定容储气罐动态模型 storage_tank_step
# ------------------------------------------------------------
# 物理意义：储气罐是定容容器，核心状态为 m_tank、T_tank、p_tank。
# 输入：上一时刻 state、进气质量流量、进气焓、出气质量流量、时间步长。
# 输出：下一时刻 state = {mass, temperature, pressure}
# 至少满足：
#   dm/dt = mdot_in - mdot_out
#   pV = mRT
# 当前实现：比等温模型更进一步，采用集中参数能量平衡：
#   U_new = U_old + mdot_in*h_in*dt - mdot_out*h_out*dt + Q_wall*dt
# ============================================================
def storage_tank_step(state, m_in, h_in, m_out, dt):
    m_old = max(state['mass'], 1e-6)
    T_old = state['temperature']
    p_old = state['pressure']
    u_old = u_from_T_p(T_old, p_old)
    h_out = h_from_T_p(T_old, p_old)
    heat_wall_w = -storage_heat_transfer_coefficient_wk * (T_old - ambient_temperature_k)

    internal_energy = (
        m_old * u_old
        + m_in * h_in * dt
        - m_out * h_out * dt
        + heat_wall_w * dt
    )
    m_new = max(m_old + (m_in - m_out) * dt, 1e-6)
    rho_new = max(m_new / storage_volume_m3, 1e-9)
    u_new = internal_energy / m_new
    T_new, p_new = T_p_from_u_rho(u_new, rho_new)
    return {'mass': m_new, 'temperature': T_new, 'pressure': p_new}

# ============================================================
# 模型 5：热储能 TES 动态模型 tes_step
# ------------------------------------------------------------
# 物理意义：TES 储存压缩热，并在放电阶段加热进入膨胀机的空气。
# 输入：上一时刻 TES 状态、充热功率、放热功率、时间步长。
# 输出：下一时刻 TES 状态 = {energy_j, temperature_k}
# 能量平衡：
#   E_new = E_old + Q_charge*dt - Q_discharge*dt - UA_loss*(T_TES - T_amb)*dt
# ============================================================
def tes_step(tes_state, heat_charge, heat_discharge, dt):
    energy_old = tes_state['energy_j']
    temperature_old = tes_state['temperature_k']
    loss_w = tes_ambient_loss_coefficient_wk * (temperature_old - ambient_temperature_k)
    energy_new = max(energy_old + heat_charge * dt - heat_discharge * dt - loss_w * dt, 0.0)
    temperature_k = ambient_temperature_k + energy_new / (tes_mass_kg * tes_specific_heat_j_kg_k)
    return {'energy_j': energy_new, 'temperature_k': temperature_k}

# ============================================================
# 辅助函数：多级压缩/膨胀的压力序列
# ------------------------------------------------------------
# 作用：在给定入口压力、出口压力和级数后，生成等压比分配的级间压力。
# ============================================================
def pressure_sequence(p_start, p_end, stages):
    ratio = (p_end / p_start) ** (1 / stages)
    values = [p_start]
    for _ in range(stages):
        values.append(values[-1] * ratio)
    values[-1] = p_end
    return values


## 9. 系统状态机与求解层代码

这里按时间步推进完整 CAES 周期。状态机包含 `charge`、`hold`、`discharge` 三类主要模式，并在达到压力约束时自动进入受限模式。


In [ ]:
# 系统状态机与求解层代码
def run_caes_cycle():
    dt = time_step_minutes * 60
    total_hours = charge_hours + hold_hours + discharge_hours
    total_steps = int(round(total_hours * 3600 / dt)) + 1

    initial_pressure_pa = initial_storage_pressure_bar * 1e5
    initial_rho = rho_from_T_p(ambient_temperature_k, initial_pressure_pa)
    storage_state = {
        'mass': initial_rho * storage_volume_m3,
        'temperature': ambient_temperature_k,
        'pressure': initial_pressure_pa
    }
    tes_state = {
        'energy_j': max(tes_mass_kg * tes_specific_heat_j_kg_k * (tes_initial_temperature_c - ambient_temperature_c), 0.0),
        'temperature_k': tes_initial_temperature_c + 273.15
    }

    time_hours = []
    pressure_bar = []
    air_temperature_c = []
    air_mass_kg = []
    tes_temperature_c = []
    compressor_power_mw = []
    expander_power_mw = []
    net_power_mw = []
    charge_heat_mw = []
    discharge_heat_mw = []
    mode_by_step = []

    electric_input_j = 0.0
    electric_output_j = 0.0
    compression_heat_recovered_j = 0.0
    tes_heat_delivered_j = 0.0

    for step in range(total_steps):
        current_hour = step * dt / 3600
        if current_hour < charge_hours:
            mode = 'charge'
        elif current_hour < charge_hours + hold_hours:
            mode = 'hold'
        else:
            mode = 'discharge'

        p_tank = storage_state['pressure']
        T_tank = storage_state['temperature']
        m_dot_in = 0.0
        m_dot_out = 0.0
        h_in_storage = 0.0
        compressor_power_w = 0.0
        expander_power_w = 0.0
        heat_charge_w = 0.0
        heat_discharge_w = 0.0

        if mode == 'charge':
            if p_tank >= 0.98 * max_storage_pressure_bar * 1e5:
                mode = 'charge_pressure_limited'
            else:
                m_dot_in = mass_flow_kg_s
                p_final = min(max(p_tank * 1.005, min_storage_pressure_bar * 1e5), 0.98 * max_storage_pressure_bar * 1e5)
                pressures = pressure_sequence(ambient_pressure_pa, p_final, compressor_stages)
                T_air = ambient_temperature_k
                p_air = ambient_pressure_pa
                tes_capacity_rate = max(tes_mass_kg * tes_specific_heat_j_kg_k / dt, 1.0)

                for stage in range(compressor_stages):
                    T_air, h_air, power_w = compressor_stage(
                        T_air,
                        pressures[stage],
                        pressures[stage + 1],
                        compressor_efficiency,
                        m_dot_in
                    )
                    compressor_power_w += power_w / motor_efficiency

                    cp_hot = air_props(T_air, pressures[stage + 1])['cp']
                    hot_capacity_rate = m_dot_in * cp_hot
                    T_air, _, q_w = cooler(
                        T_air,
                        tes_state['temperature_k'],
                        hot_capacity_rate,
                        tes_capacity_rate,
                        heat_exchanger_effectiveness
                    )
                    heat_charge_w += q_w

                h_in_storage = h_from_T_p(T_air, p_final)
                compression_heat_recovered_j += heat_charge_w * dt
                electric_input_j += compressor_power_w * dt

        elif mode == 'discharge':
            if p_tank <= 1.05 * min_storage_pressure_bar * 1e5 or storage_state['mass'] <= 1e-6:
                mode = 'discharge_pressure_limited'
            else:
                available_mass_flow = max((storage_state['mass'] - 1e-6) / dt, 0.0)
                m_dot_out = min(mass_flow_kg_s, available_mass_flow)
                T_air = T_tank
                p_start = p_tank
                p_final = ambient_pressure_pa

                cp_current = air_props(T_air, p_start)['cp']
                target_turbine_inlet_k = min(
                    max_turbine_inlet_temperature_c + 273.15,
                    max(tes_state['temperature_k'] - minimum_tes_approach_temperature_k, T_air)
                )
                preheat_w = max(m_dot_out * cp_current * (target_turbine_inlet_k - T_air), 0.0)
                available_tes_power = max(tes_state['energy_j'] / dt, 0.0)
                preheat_w = min(preheat_w, available_tes_power)
                if preheat_w > 0:
                    T_air += preheat_w / max(m_dot_out * cp_current, 1e-9)
                    heat_discharge_w += preheat_w

                pressures = pressure_sequence(p_start, p_final, expander_stages)
                for stage in range(expander_stages):
                    T_air, h_air, power_w = expander_stage(
                        T_air,
                        pressures[stage],
                        pressures[stage + 1],
                        expander_efficiency,
                        m_dot_out
                    )
                    expander_power_w += power_w * generator_efficiency

                    if stage < expander_stages - 1:
                        cp_reheat = air_props(T_air, pressures[stage + 1])['cp']
                        target_reheat_k = min(
                            max_turbine_inlet_temperature_c + 273.15,
                            max(tes_state['temperature_k'] - minimum_tes_approach_temperature_k, T_air)
                        )
                        reheat_w = max(m_dot_out * cp_reheat * (target_reheat_k - T_air), 0.0)
                        available_tes_power = max((tes_state['energy_j'] - heat_discharge_w * dt) / dt, 0.0)
                        reheat_w = min(reheat_w, available_tes_power)
                        if reheat_w > 0:
                            T_air += reheat_w / max(m_dot_out * cp_reheat, 1e-9)
                            heat_discharge_w += reheat_w

                electric_output_j += expander_power_w * dt
                tes_heat_delivered_j += heat_discharge_w * dt

        if mode.startswith('charge'):
            storage_state = storage_tank_step(storage_state, m_dot_in, h_in_storage, 0.0, dt)
            tes_state = tes_step(tes_state, heat_charge_w, 0.0, dt)
        elif mode.startswith('discharge'):
            storage_state = storage_tank_step(storage_state, 0.0, 0.0, m_dot_out, dt)
            tes_state = tes_step(tes_state, 0.0, heat_discharge_w, dt)
        else:
            storage_state = storage_tank_step(storage_state, 0.0, 0.0, 0.0, dt)
            tes_state = tes_step(tes_state, 0.0, 0.0, dt)

        time_hours.append(current_hour)
        pressure_bar.append(storage_state['pressure'] / 1e5)
        air_temperature_c.append(storage_state['temperature'] - 273.15)
        air_mass_kg.append(storage_state['mass'])
        tes_temperature_c.append(tes_state['temperature_k'] - 273.15)
        compressor_power_mw.append(compressor_power_w / 1e6)
        expander_power_mw.append(expander_power_w / 1e6)
        net_power_mw.append((expander_power_w - compressor_power_w) / 1e6)
        charge_heat_mw.append(heat_charge_w / 1e6)
        discharge_heat_mw.append(heat_discharge_w / 1e6)
        mode_by_step.append(mode)

    round_trip_efficiency = electric_output_j / electric_input_j if electric_input_j > 0 else 0.0
    return {
        'time_hours': np.array(time_hours),
        'pressure_bar': np.array(pressure_bar),
        'air_temperature_c': np.array(air_temperature_c),
        'air_mass_kg': np.array(air_mass_kg),
        'tes_temperature_c': np.array(tes_temperature_c),
        'compressor_power_mw': np.array(compressor_power_mw),
        'expander_power_mw': np.array(expander_power_mw),
        'net_power_mw': np.array(net_power_mw),
        'charge_heat_mw': np.array(charge_heat_mw),
        'discharge_heat_mw': np.array(discharge_heat_mw),
        'mode_by_step': mode_by_step,
        'electric_input_j': electric_input_j,
        'electric_output_j': electric_output_j,
        'compression_heat_recovered_j': compression_heat_recovered_j,
        'tes_heat_delivered_j': tes_heat_delivered_j,
        'round_trip_efficiency': round_trip_efficiency
    }

results = run_caes_cycle()

time_hours = results['time_hours']
pressure_bar = results['pressure_bar']
air_temperature_c = results['air_temperature_c']
air_mass_kg = results['air_mass_kg']
tes_temperature_c = results['tes_temperature_c']
compressor_power_mw = results['compressor_power_mw']
expander_power_mw = results['expander_power_mw']
net_power_mw = results['net_power_mw']
charge_heat_mw = results['charge_heat_mw']
discharge_heat_mw = results['discharge_heat_mw']
mode_by_step = results['mode_by_step']
round_trip_efficiency = results['round_trip_efficiency']

print(f'仿真步数: {len(time_hours)}')
print(f'最高储气压力: {np.max(pressure_bar):.2f} bar')
print(f'最低储气压力: {np.min(pressure_bar):.2f} bar')
print(f'往返效率: {round_trip_efficiency:.2%}')




## 10. 结果可视化代码

这里绘制储气压力、储气温度、储气质量、TES 温度、压缩/膨胀/净功率和运行模式时间轴。


In [ ]:
# 结果可视化代码
mode_order = [
    'charge',
    'charge_pressure_limited',
    'hold',
    'discharge',
    'discharge_pressure_limited'
]
mode_to_value = {name: index for index, name in enumerate(mode_order)}
mode_values = [mode_to_value.get(mode, -1) for mode in mode_by_step]

fig, axes = plt.subplots(3, 2, figsize=(16, 12), constrained_layout=True)

axes[0, 0].plot(time_hours, pressure_bar, color='#2868a6', linewidth=2)
axes[0, 0].axhline(max_storage_pressure_bar, color='#a83232', linestyle='--', linewidth=1, label='最高压力')
axes[0, 0].axhline(min_storage_pressure_bar, color='#555555', linestyle='--', linewidth=1, label='最低压力')
axes[0, 0].set_title('储气罐压力动态')
axes[0, 0].set_xlabel('时间 / h')
axes[0, 0].set_ylabel('压力 / bar')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.25)

axes[0, 1].plot(time_hours, air_temperature_c, color='#cc6f2d', linewidth=2)
axes[0, 1].axhline(ambient_temperature_c, color='#555555', linestyle='--', linewidth=1, label='环境温度')
axes[0, 1].set_title('储气罐空气温度')
axes[0, 1].set_xlabel('时间 / h')
axes[0, 1].set_ylabel('温度 / degC')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.25)

axes[1, 0].plot(time_hours, air_mass_kg, color='#39704f', linewidth=2)
axes[1, 0].set_title('储气罐空气质量')
axes[1, 0].set_xlabel('时间 / h')
axes[1, 0].set_ylabel('质量 / kg')
axes[1, 0].grid(True, alpha=0.25)

axes[1, 1].plot(time_hours, tes_temperature_c, color='#b23b45', linewidth=2)
axes[1, 1].axhline(ambient_temperature_c, color='#555555', linestyle='--', linewidth=1, label='环境温度')
axes[1, 1].set_title('TES 等效温度')
axes[1, 1].set_xlabel('时间 / h')
axes[1, 1].set_ylabel('温度 / degC')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.25)

axes[2, 0].plot(time_hours, compressor_power_mw, color='#7a4ea3', linewidth=2, label='压缩耗电')
axes[2, 0].plot(time_hours, expander_power_mw, color='#1f7a4d', linewidth=2, label='膨胀发电')
axes[2, 0].plot(time_hours, net_power_mw, color='#222222', linewidth=1.5, label='净功率')
axes[2, 0].set_title('系统功率动态')
axes[2, 0].set_xlabel('时间 / h')
axes[2, 0].set_ylabel('功率 / MW')
axes[2, 0].legend()
axes[2, 0].grid(True, alpha=0.25)

axes[2, 1].step(time_hours, mode_values, where='post', color='#36454f', linewidth=2)
axes[2, 1].set_title('运行模式时间轴')
axes[2, 1].set_xlabel('时间 / h')
axes[2, 1].set_yticks(list(mode_to_value.values()))
axes[2, 1].set_yticklabels(list(mode_to_value.keys()))
axes[2, 1].grid(True, axis='x', alpha=0.25)

plt.show()


## 11. 关键结果输出

这里集中输出电输入、电输出、往返效率、储气压力范围、TES 温度变化和热量利用情况。


In [ ]:
# 关键结果输出
electric_input_mwh = results['electric_input_j'] / 3.6e9
electric_output_mwh = results['electric_output_j'] / 3.6e9
compression_heat_recovered_mwh = results['compression_heat_recovered_j'] / 3.6e9
tes_heat_delivered_mwh = results['tes_heat_delivered_j'] / 3.6e9

summary_rows = [
    {'指标': '压缩阶段电输入', '数值': f'{electric_input_mwh:.3f}', '单位': 'MWh'},
    {'指标': '膨胀阶段电输出', '数值': f'{electric_output_mwh:.3f}', '单位': 'MWh'},
    {'指标': '系统往返效率', '数值': f'{round_trip_efficiency * 100:.2f}', '单位': '%'},
    {'指标': '最高储气压力', '数值': f'{np.max(pressure_bar):.2f}', '单位': 'bar'},
    {'指标': '最低储气压力', '数值': f'{np.min(pressure_bar):.2f}', '单位': 'bar'},
    {'指标': '最终储气压力', '数值': f'{pressure_bar[-1]:.2f}', '单位': 'bar'},
    {'指标': '最高储气温度', '数值': f'{np.max(air_temperature_c):.2f}', '单位': 'degC'},
    {'指标': '最终 TES 温度', '数值': f'{tes_temperature_c[-1]:.2f}', '单位': 'degC'},
    {'指标': '压缩热回收量', '数值': f'{compression_heat_recovered_mwh:.3f}', '单位': 'MWh_th'},
    {'指标': 'TES 放热量', '数值': f'{tes_heat_delivered_mwh:.3f}', '单位': 'MWh_th'},
]

if Markdown is not None:
    display(Markdown(markdown_table(summary_rows, ['指标', '数值', '单位'])))
else:
    print(markdown_table(summary_rows, ['指标', '数值', '单位']))

mode_counts = {mode: mode_by_step.count(mode) for mode in sorted(set(mode_by_step))}
print('运行模式步数:', mode_counts)


## 12. 结果分析提示

1. 观察储气压力曲线：充电阶段压力应上升，放电阶段压力应下降；若很快触及最高或最低压力，说明储气容积、质量流量和运行时长不匹配。
2. 观察 TES 温度曲线：充电阶段 TES 应吸收压缩热升温，放电阶段向空气放热降温。
3. 对比压缩耗电和膨胀发电：往返效率受压缩机效率、膨胀机效率、换热器有效度和 TES 容量共同影响。
4. 调大 `storage_volume_m3` 后，压力变化会变慢，放电持续能力增强，但单位时间压力提升也会下降。
5. 调高 `heat_exchanger_effectiveness` 后，压缩热回收更充分，TES 温度更高，膨胀前再热能力更强。
6. 调大 `tes_mass_kg` 会增加储热惯性；它可能提高放电稳定性，但过大时温升变小，需要结合换热器有效度分析。
7. 后续集成到系统时，可以把参数层代码绑定到侧边栏，实现“参数修改 -> 代码变化 -> 下游自动重算 -> 图像更新”的代码可视化流程。
